# Hype Clipper ? Hook + Payoff Shorts Generator (Max 50s)
Select your AI model in **Step 2** and run **Step 3**.

In [ ]:
#@title Step 1: Mount Drive, Clone Repo & Install Ollama
from google.colab import drive
import os, shutil, subprocess, time

drive.mount('/content/drive')
!curl -fsSL https://deno.land/install.sh | sh > /dev/null 2>&1 || true
os.environ['PATH'] += ':/root/.deno/bin:/usr/local/bin'

!rm -rf hype-clipper && git clone https://github.com/Aromalajith8/hype-clipper.git hype-clipper
!pip install -q -r hype-clipper/requirements.txt

if not shutil.which('ollama'):
    !apt-get -qq update && apt-get -qq install -y zstd curl > /dev/null 2>&1
    !curl -fsSL https://ollama.com/install.sh | sh || true

if not shutil.which('ollama'):
    print('Fallback: downloading Ollama Linux amd64 release directly...')
    !curl -fsSL -L https://github.com/ollama/ollama/releases/latest/download/ollama-linux-amd64.tgz -o /tmp/ollama.tgz
    !mkdir -p /usr/local && tar -xzf /tmp/ollama.tgz -C /usr/local
    !rm -f /tmp/ollama.tgz

ollama_bin = shutil.which('ollama') or '/usr/local/bin/ollama'
assert os.path.exists(ollama_bin), 'Ollama binary could not be installed in this runtime.'

subprocess.Popen([ollama_bin, 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(4)
print('Step 1 complete! Ollama binary:', ollama_bin)

In [ ]:
#@title Step 2: Choose Model & Stream Settings
STREAM_URL = '' #@param {type:"string"}
AI_PROVIDER = 'ollama' #@param ['ollama', 'claude', 'auto', 'groq', 'gemini', 'heuristic']
OLLAMA_MODEL = 'qwen3.5:9b' #@param ['qwen3.5:9b', 'gemma4:12b', 'qwen3.5:27b', 'gemma4:26b']
CLAUDE_MODEL = 'claude-fable-5-2' #@param ['claude-fable-5-2', 'claude-opus-5-5']
ANTHROPIC_API_KEY = '' #@param {type:"string"}
CLIPS = 40 #@param {type:"slider", min:5, max:100, step:5}
MIN_LEN = 25 #@param {type:"slider", min:15, max:45, step:5}
MAX_LEN = 50 #@param {type:"slider", min:20, max:50, step:5}
CHAT_BOX_SIDE = 'right' #@param ['right', 'left', 'none']
SPLIT_CAM = False #@param {type:"boolean"}
OUTPUT_FOLDER = '/content/drive/MyDrive/HypeClips' #@param {type:"string"}
COOKIES_PATH = '/content/drive/MyDrive/cookies.txt' #@param {type:"string"}

In [ ]:
#@title Step 3: Pull Model & Generate Clips
import os, shutil, subprocess

assert STREAM_URL.strip(), 'Paste a stream URL in STREAM_URL inside Step 2 first!'

if ANTHROPIC_API_KEY.strip():
    os.environ['ANTHROPIC_API_KEY'] = ANTHROPIC_API_KEY.strip()

ollama_bin = shutil.which('ollama') or '/usr/local/bin/ollama'
if AI_PROVIDER in ('ollama', 'auto') and not os.environ.get('ANTHROPIC_API_KEY'):
    print(f'Pulling {OLLAMA_MODEL}...')
    subprocess.run([ollama_bin, 'pull', OLLAMA_MODEL], check=False)

cmd = [
    'python3', 'hype-clipper/clipper.py', STREAM_URL.strip(),
    '--clips', str(CLIPS),
    '--min-len', str(MIN_LEN),
    '--max-len', str(min(MAX_LEN, 50)),
    '--chat-box', CHAT_BOX_SIDE,
    '--out', OUTPUT_FOLDER,
    '--ai-provider', AI_PROVIDER,
    '--claude-model', CLAUDE_MODEL,
    '--ollama-model', OLLAMA_MODEL,
]

if SPLIT_CAM:
    cmd.append('--split-cam')
if os.path.exists(COOKIES_PATH):
    cmd += ['--cookies', COOKIES_PATH]

subprocess.run(cmd)